# Week 03 Adapter Sprint: QLoRA smoke test (Colab T4)

**Purpose:** an *execution-validation checkpoint*. This is **not** the SFT experiment and **not** the final assignment notebook.

It checks, on real T4 hardware, that:
- the pinned stack installs
- bitsandbytes NF4 loading works
- LoRA attaches to the verified q/k/v/o targets
- assistant-only masking still holds
- gradients reach LoRA
- fp16 loss stays finite for **5 optimizer steps**
- memory and timing instrumentation work
- one real `smoke` row is logged
- the adapter saves

All training logic lives in `training/train_qlora.py`. This notebook only calls repository scripts.

**Rules:**
- If any cell fails, **stop**. Do not change settings and re-run.
- Record the failure (see Section 10). Failed CUDA runs are logged automatically.
- No numbers in this notebook are pre-filled. Everything shown comes from this execution.

## 1. Runtime / GPU preflight (before installing anything)
Requires **Runtime → Change runtime type → T4 GPU**. This cell also records Colab's preinstalled torch version, so the install step can be constrained to keep it.

In [ ]:
!nvidia-smi
import sys, torch
print("python", sys.version.split()[0])
print("preinstalled torch", torch.__version__, "| built for CUDA", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("STOP: no CUDA GPU. Runtime > Change runtime type > T4 GPU, then restart from Section 1.")
print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
with open("/content/torch_constraint.txt", "w") as f:
    f.write(f"torch=={torch.__version__}\n")
print(open("/content/torch_constraint.txt").read())

## 2. Clone the repository
Set `REPO_URL` to your GitHub repository. Leave it public, or clone with a token from Colab **Secrets**. **Never paste a token into this notebook.**

In [ ]:
REPO_URL = "https://github.com/<your-github-username>/week03-adapter-sprint.git"   # EDIT ME
REPO_DIR = "/content/week03-adapter-sprint"

import os
if "<your-github-username>" in REPO_URL:
    raise RuntimeError("STOP: set REPO_URL first.")
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -5

## 3. Install pinned dependencies
- Installs from `requirements.txt`, which has exact pins for transformers, peft, trl, bitsandbytes, accelerate and datasets.
- The constraint file **pins torch to Colab's preinstalled CUDA build**. If any package would need a different torch, pip **fails** instead of silently replacing it.
- Nothing is upgraded beyond the pins.

In [ ]:
%cd /content/week03-adapter-sprint
!pip install -r requirements.txt -c /content/torch_constraint.txt
!pip check || echo "pip check reported issues (see above) - review before continuing"

## 4. ⚠️ RESTART REQUIRED
**Runtime → Restart session** now. bitsandbytes and the newly installed transformers must be imported fresh.

After restarting, **continue from Section 5**. Do **not** re-run Sections 1–3.

## 5. Environment + version verification
Runs `training/preflight.py --require-cuda`, which checks:
- installed versions against `requirements.txt` (any mismatch fails)
- the torch/CUDA build and GPU compute capability (the compiled kernels must include `sm_75`)
- an fp16 matmul
- a real NF4 + double-quant bitsandbytes round trip on the GPU
- the git commit (the tree must be clean and must contain Step 3 commit `60a1789`)
- the frozen dataset hashes

Optionally set `EXPECTED_COMMIT` to the exact commit you pushed for this smoke test.

In [ ]:
import subprocess

def run(cmd):
    """Run a repository script, stream its output, and STOP the notebook on a non-zero exit."""
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"STOP: command failed with exit code {rc}: {' '.join(cmd)}")

import os, sys
os.chdir("/content/week03-adapter-sprint")
EXPECTED_COMMIT = ""   # optional, e.g. the short hash of the smoke-harness commit

cmd = [sys.executable, "training/preflight.py", "--require-cuda"]
if EXPECTED_COMMIT:
    cmd += ["--expected-commit", EXPECTED_COMMIT]
run(cmd)

## 6. Dataset / config integrity (no regeneration)
- Runs the full dataset audit, including a byte-identical regeneration check into a *temporary* directory.
- Confirms `data/` has no uncommitted changes.
- The committed dataset is never rewritten.

In [ ]:
run([sys.executable, "scripts/audit_instruction_data.py"])
run(["git", "diff", "--exit-code", "--stat", "--", "data/"])
print("data/ matches the committed frozen dataset")

## 7. Masking sanity check (real records, tokenizer only)
- `check_masking.py` prints the prompt boundary and the trainable assistant region for 3 real records, including null-address ones.
- It then encodes all 400 records with the fail-fast checks.
- `--dry-run` repeats the config, data and masking checks through the same entry point used for training.

In [ ]:
run([sys.executable, "training/check_masking.py"])
run([sys.executable, "training/train_qlora.py", "--run-label", "smoke", "--max-steps", "5", "--dry-run"])

## 8. 5-step QLoRA smoke run
This uses the approved baseline: NF4 + double quantization, fp16 compute, r=16, α=32, dropout 0.05, q/k/v/o targets, gradient checkpointing on, micro-batch 4 × accumulation 4 (effective 16), max sequence length 512, and `auto_find_batch_size=False`.

Smoke-only settings:

| Setting | Value | Why |
|---|---|---|
| `--max-steps` | 5 | overrides the epoch count; the cosine/warmup schedule is computed over 5 steps (1 warmup step) |
| `--logging-steps` | 1 | shows the loss of every step |
| `--eval-steps` | 5 | runs the eval path once |
| output directory | `outputs/smoke_adapter/<run_id>/` | kept separate from the production SFT output |
| method in the log | `qlora_sft_smoke` | distinguishable from the full SFT run |

Look in the output for: `QUANTIZATION EVIDENCE`, `LINEAR / PROJECTION MODULES`, `PARAMETERS`, `GRADIENT CHECK`, per-step `loss`, and `RUN RESULT`.

**If this cell fails: STOP.** Do not change batch size or other settings and re-run. The failed run is already logged. Go to Section 10.

In [ ]:
run([sys.executable, "training/train_qlora.py",
     "--run-label", "smoke", "--max-steps", "5", "--logging-steps", "1", "--eval-steps", "5"])

## 9. Inspect the experiment log and smoke outputs
Everything below is read from files written by the run.

In [ ]:
import glob, json, os
import pandas as pd

log = pd.read_csv("logs/experiment_log.csv")
print(f"{len(log)} row(s) in logs/experiment_log.csv")
smoke = log[log["run_label"] == "smoke"]
display(smoke.tail(1).T)

runs = sorted(glob.glob("outputs/smoke_adapter/qlora_sft_smoke-*"))
if not runs:
    print("No smoke output directory found (the run may have failed before saving).")
else:
    latest = runs[-1]
    print("latest smoke output:", latest)
    adapter = os.path.join(latest, "final_adapter")
    print("adapter files:", sorted(os.listdir(adapter)) if os.path.isdir(adapter) else "MISSING")
    hist = json.load(open(os.path.join(latest, "log_history.json")))
    rows = [{"step": h.get("step"), "loss": h.get("loss"), "grad_norm": h.get("grad_norm"),
             "learning_rate": h.get("learning_rate"), "eval_loss": h.get("eval_loss")}
            for h in hist if "loss" in h or "eval_loss" in h]
    display(pd.DataFrame(rows))

!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
print("Note: nvidia-smi 'memory.used' includes the CUDA context and is NOT the logged torch peak.")
!git status --short
!git diff --stat

## 10. Smoke-test conclusion (fill in by hand from the output above; do not guess)

| Check | Evidence (cell/section) | Result |
|---|---|---|
| Pinned stack installed, versions match | §5 preflight | |
| T4 visible, `sm_75` kernels present | §5 | |
| bitsandbytes NF4 + double-quant kernels work | §5 | |
| Frozen data unchanged | §6 | |
| Masking checks pass | §7 | |
| All `*_proj` are `Linear4bit`, `quant_type=nf4`, nested=True | §8 `QUANTIZATION EVIDENCE` | |
| LoRA on q/k/v/o, 96 modules, 2,162,688 trainable | §8 `PARAMETERS` | |
| Gradients reach LoRA, none on base (expect `lora_nonzero_grad` = 96 of 192 LoRA tensors at the first finite step: LoRA initializes B=0, so A's gradient is exactly zero until B has moved) | §8 `GRADIENT CHECK` | |
| Loss finite at all 5 logged steps | §8 / §9 | |
| 5 optimizer steps completed | §9 `total_steps` | |
| Peak allocated / reserved GiB recorded | §9 | |
| Wall clock + avg step time recorded | §9 | |
| One `smoke` row logged | §9 | |
| Adapter saved under `outputs/smoke_adapter/` | §9 | |

**If a step failed**, record:
- failure type and full error (from §8)
- configuration
- likely failure class
- evidence
- **one** proposed corrective action

Do not retry with different settings until it has been reviewed.

**Persist the log row.** Colab storage is temporary. Either commit `logs/experiment_log.csv` from here (only that file should show in `git diff --stat`), or download it:
```python
from google.colab import files; files.download("logs/experiment_log.csv")
```
The smoke adapter is throwaway and git-ignored. It is **never** used as the SFT model.